In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
#Import all needed lib
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score,f1_score
import numpy as np




In [ ]:
# Task 1: Write your code here:
_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_values = df.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])



In [ ]:
# Fill missing values with mean for each column
df = df.fillna(df.mean())
print(df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:

def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols)) # no Categorical Columns

In [ ]:
# Task 4: Write your code here:
# in real life we split first then we scale so i wiil do that :)
# Define features and target

X = df.drop('Target', axis=1)
y = df['Target']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Target'].dropna(), bins=30, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show() #as shown the target is imbalanced

In [ ]:
# Task 1: Write your code here:
#done above

In [ ]:
# Task 2,3,4,5: Write your code here:
lr_accuracy = []
lr_f1 = []

n_splits = 5

# Stratified becuse the target is imbalnced

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)
    model.fit(X_train, y_train)

    #Predict
    y_pred = model.predict(X_test)


    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_f1.append(f1)


In [ ]:
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_cols = df.drop(columns=['Target'],inplace=True)
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: